<a href="https://colab.research.google.com/github/GodishalaAshwith/DeepLearning/blob/main/DLInt2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

List of Programs for Lab Internal 2

1. pre-trained models LeNet, AlexNet, ZF-Net, VGGNet, Google Inception Module, Resnet.
2. Visualizing Convolutional Neural Networks
3. Guided Backpropagation to discover which input pixels influence the perceptron.
4. Implement Auto Encoder model on MINIST dataset.
5. Implement Undercomplete AE and Overcomplete AE  
6. Implement Regularization in AE  
7. Implement denoising AE  
8. Demonstrate PCA with AE on a dataset.
9. Implement Sparse AE and Contractive AE.
10. Implement RNN for predicting the next character
11. Implement BERT model  
12. Implement GAN model on MNIST.

In [ ]:
#https://colab.research.google.com/drive/1hT_b5ovuJL7Eztai_3Zb0y2iSeOkXGEX?usp=sharing

### pre-trained models LeNet, AlexNet, ZF-Net, VGGNet, Google Inception Module, Resnet.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# ================= DATA =================
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize and reshape for CNN
x_train, x_test = x_train/255.0, x_test/255.0
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)


# ================= LeNet-5 (Simplified) =================
# Original: tanh + subsampling | Here: ReLU + pooling for modern practice
lenet_model = models.Sequential([
    layers.Conv2D(6,5,activation='relu',padding='same',input_shape=(28,28,1)),
    layers.AveragePooling2D(2),
    layers.Conv2D(16,5,activation='relu'),
    layers.AveragePooling2D(2),
    layers.Flatten(),
    layers.Dense(120,activation='relu'),
    layers.Dense(84,activation='relu'),
    layers.Dense(10,activation='softmax')
])

lenet_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
lenet_model.fit(x_train,y_train,epochs=2,verbose=0)
print("LeNet Accuracy:", lenet_model.evaluate(x_test,y_test,verbose=0)[1])


# ================= AlexNet (Simplified) =================
# Original uses large kernels (11x11), dropout, LRN; adapted for 28x28 input
alexnet_model = models.Sequential([
    layers.Conv2D(64,3,activation='relu',padding='same',input_shape=(28,28,1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(128,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10,activation='softmax')
])

alexnet_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
alexnet_model.fit(x_train,y_train,epochs=2,verbose=0)
print("AlexNet Accuracy:", alexnet_model.evaluate(x_test,y_test,verbose=0)[1])


# ================= ZF-Net (Simplified) =================
# Improvement over AlexNet: smaller stride & better feature capture
zf_model = models.Sequential([
    layers.Conv2D(64,3,strides=1,activation='relu',padding='same',input_shape=(28,28,1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(128,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(256,activation='relu'),
    layers.Dense(10,activation='softmax')
])

zf_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
zf_model.fit(x_train,y_train,epochs=2,verbose=0)
print("ZF-Net Accuracy:", zf_model.evaluate(x_test,y_test,verbose=0)[1])


# ================= VGG (Mini VGG) =================
# VGG uses stacked 3x3 conv layers; depth reduced for MNIST
vgg_model = models.Sequential([
    layers.Conv2D(32,3,activation='relu',padding='same',input_shape=(28,28,1)),
    layers.Conv2D(32,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),

    layers.Conv2D(64,3,activation='relu',padding='same'),
    layers.Conv2D(64,3,activation='relu',padding='same'),
    layers.MaxPooling2D(2),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(10,activation='softmax')
])

vgg_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
vgg_model.fit(x_train,y_train,epochs=1,verbose=0)
print("VGG Accuracy:", vgg_model.evaluate(x_test,y_test,verbose=0)[1])


# ================= Inception (Simplified Module) =================x`
# Parallel filters: 1x1, 3x3, and pooling branches
inp = layers.Input(shape=(28,28,1))

c1 = layers.Conv2D(16,1,activation='relu',padding='same')(inp)
c2 = layers.Conv2D(16,3,activation='relu',padding='same')(inp)
c3 = layers.MaxPooling2D(3,strides=1,padding='same')(inp)

x = layers.concatenate([c1,c2,c3])
x = layers.Flatten()(x)
out = layers.Dense(10,activation='softmax')(x)

inception_model = models.Model(inp,out)
inception_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
inception_model.fit(x_train,y_train,epochs=1,verbose=0)
print("Inception Accuracy:", inception_model.evaluate(x_test,y_test,verbose=0)[1])


# ================= ResNet (Basic Residual Block) =================
# Skip connection helps avoid vanishing gradient
inp = layers.Input(shape=(28,28,1))

x = layers.Conv2D(32,3,padding='same',activation='relu')(inp)
skip = x

x = layers.Conv2D(32,3,padding='same')(x)
x = layers.Add()([x,skip])
x = layers.Activation('relu')(x)

x = layers.Flatten()(x)
out = layers.Dense(10,activation='softmax')(x)

resnet_model = models.Model(inp,out)
resnet_model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])
resnet_model.fit(x_train,y_train,epochs=1,verbose=0)
print("ResNet Accuracy:", resnet_model.evaluate(x_test,y_test,verbose=0)[1])

### Visualizing Convolutional Neural Networks

In [ ]:
import torch, matplotlib.pyplot as plt
from torchvision import datasets, transforms
import torch.nn as nn

# Load one MNIST image
img,_ = datasets.MNIST('./data', train=True, download=True,
                       transform=transforms.ToTensor())[0]

# Conv + Pool
x = img.unsqueeze(0)
conv = nn.Conv2d(1,8,3,padding=1)(x)
pool = nn.MaxPool2d(2)(conv)

# Plot
def show(t):
    t = t.squeeze().detach()
    for i in range(t.shape[0]):
        plt.imshow(t[i])
    plt.show()

show(conv)
show(pool)

### Guided Backpropogation

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import matplotlib.pyplot as plt

# Load + preprocess
(x, y), _ = mnist.load_data()
x = x.astype("float32")/255.0
x = x[..., None]

# Simple CNN
model = models.Sequential([
    layers.Conv2D(8, 3, activation='relu', input_shape=(28,28,1)),
    layers.Flatten(),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(x, y, epochs=1, verbose=0)

# Pick one sample
img = tf.convert_to_tensor(x[:1])

# Guided Backprop (positive gradients + positive activations)
with tf.GradientTape() as tape:
    tape.watch(img)
    preds = model(img)
    class_idx = tf.argmax(preds[0])
    loss = preds[:, class_idx]

grads = tape.gradient(loss, img)

# Guided filtering (true idea)
guided = tf.nn.relu(grads) * tf.cast(img > 0, tf.float32)

# Visualize
plt.imshow(guided[0,:,:,0])
plt.title("Guided Backpropagation")
plt.show()

### Implement Auto Encoder model on MINIST dataset.

In [ ]:
# 4. Implement Auto Encoder model on MINIST dataset
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# data
(x_train, _), _ = mnist.load_data()
x_train = x_train/255.0
x_train = x_train.reshape(-1,28*28)

print("Data shape:", x_train.shape)

# autoencoder
model = models.Sequential([
    layers.Dense(64,activation='relu',input_shape=(784,)),
    layers.Dense(784,activation='sigmoid')
])

model.compile(optimizer='adam', loss='mse')

# train
model.fit(x_train,x_train,epochs=2,verbose=0)

print("Training done")

# test reconstruction
out = model.predict(x_train[:1])

print("Input shape:", x_train[:1].shape)
print("Output shape:", out.shape)
print("Sample output values:", out[0][:5])

### Implement Undercomplete AE and Overcomplete AE

In [ ]:
# 5. Implement Undercomplete AE and Overcomplete AE
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist

# data
(x_train, _), _ = mnist.load_data()
x_train = x_train/255.0
x_train = x_train.reshape(-1,784)

print("Data loaded:", x_train.shape)

# -------- Undercomplete AE --------
model = models.Sequential([
    layers.Dense(32,activation='relu',input_shape=(784,)),  # smaller
    layers.Dense(784,activation='sigmoid')
])

model.compile(optimizer='adam',loss='mse')
model.fit(x_train,x_train,epochs=2,verbose=0)

out1 = model.predict(x_train[:1])
print("Undercomplete output shape:", out1.shape)
print("Sample output values:", out1[0][:5])


# -------- Overcomplete AE --------
model = models.Sequential([
    layers.Dense(1024,activation='relu',input_shape=(784,)),  # larger
    layers.Dense(784,activation='sigmoid')
])

model.compile(optimizer='adam',loss='mse')
model.fit(x_train,x_train,epochs=2,verbose=0)

out2 = model.predict(x_train[:1])
print("Overcomplete output shape:", out2.shape)
print("Sample output values:", out2[0][:5])

### Implement Regularization in AE

In [ ]:
# 6. Implement Regularization in AE
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import mnist

# data
(x_train, _), _ = mnist.load_data()
x_train = x_train/255.0
x_train = x_train.reshape(-1,784)

print("Data shape:", x_train.shape)

# regularized AE
model = models.Sequential([
    layers.Dense(64,activation='relu',
                 activity_regularizer=regularizers.l1(1e-4),
                 input_shape=(784,)),
    layers.Dense(784,activation='sigmoid')
])

model.compile(optimizer='adam',loss='mse')

model.fit(x_train,x_train,epochs=2,verbose=0)

print("Training complete")

# test
out = model.predict(x_train[:1])

print("Output shape:", out.shape)
print("Sample values:", out[0][:5])

### Implement denoising AE

In [ ]:
# 7. Implement denoising AE

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
import numpy as np

# data
(x_train, _), _ = mnist.load_data()
x_train = x_train/255.0
x_train = x_train.reshape(-1,784)

print("Original data shape:", x_train.shape)

# add noise
noise = 0.5 * np.random.normal(size=x_train.shape)
x_noisy = x_train + noise
x_noisy = tf.clip_by_value(x_noisy,0.,1.)

print("Noise added")

# model
model = models.Sequential([
    layers.Input(shape=(784,)), # Explicit Input layer
    layers.Dense(64,activation='relu'),
    layers.Dense(784,activation='sigmoid')
])

model.compile(optimizer='adam',loss='mse')

model.fit(x_noisy,x_train,epochs=2,verbose=0)

print("Training done")

# test
out = model.predict(x_noisy[:1])

print("Noisy input sample:", x_noisy[0][:5])
print("Reconstructed sample:", out[0][:5])


import matplotlib.pyplot as plt

imgs = [x_train[0], x_noisy[0].numpy(), out[0]]
titles = ["Original", "Noisy", "Reconstructed"]

for i in range(3):
    plt.subplot(1,3,i+1)
    plt.imshow(imgs[i].reshape(28,28))
    plt.title(titles[i])
    # plt.axis('off')

plt.show()

### PCA and AE

In [ ]:
import tensorflow as tf, matplotlib.pyplot as plt
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from sklearn.decomposition import PCA

# Data
(x,_),_ = mnist.load_data()
x = (x/255.0).reshape(-1,784)

# PCA
pca = PCA(32)
pca_rec = pca.inverse_transform(pca.fit_transform(x))

# Autoencoder
inp = layers.Input((784,))
out = layers.Dense(784, activation='sigmoid')(layers.Dense(32, activation='relu')(inp))
ae = models.Model(inp, out)
ae.compile('adam','mse')
ae.fit(x,x,epochs=2,verbose=0)
ae_rec = ae.predict(x[:1])

# Show
imgs = [x[0], pca_rec[0], ae_rec[0]]
titles = ["Original","PCA","AE"]

for i in range(3):
    plt.subplot(1,3,i+1)
    plt.imshow(imgs[i].reshape(28,28))
    plt.title(titles[i]); plt.axis('off')

plt.show()

### Implement Sparse AE and Contractive AE.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.datasets import mnist

# Data
(x,_),_ = mnist.load_data()
x = (x/255.0).reshape(-1,784).astype("float32")

# Sparse AE
s = models.Sequential([
    layers.Dense(64,activation='relu',
                 activity_regularizer=regularizers.l1(1e-4),
                 input_shape=(784,)),
    layers.Dense(784,activation='sigmoid')
])
s.compile('adam','mse'); s.fit(x,x,epochs=2,verbose=0)

# Contractive AE
inp = tf.keras.Input((784,))
enc_layer = layers.Dense(64,activation='relu')
enc = enc_layer(inp)
out = layers.Dense(784,activation='sigmoid')(enc)
c = models.Model(inp,out)

opt = tf.keras.optimizers.Adam()

for _ in range(2):
    with tf.GradientTape() as tape:
        y = c(x, training=True)   # IMPORTANT: forward pass inside tape
        loss = tf.reduce_mean((x - y)**2)
        penalty = 1e-4 * tf.reduce_mean(enc_layer.kernel**2)
        total = loss + penalty

    grads = tape.gradient(total, c.trainable_weights)
    opt.apply_gradients(zip(grads, c.trainable_weights))

print("Sparse:", s.predict(x[:1]).shape,
      "Contractive:", c.predict(x[:1]).shape)

### RNN

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Data
chars = ['a','b','c','d']
X = np.eye(4)[[0,1,2]].reshape(3,1,4)
y = np.eye(4)[[1,2,3]]

# Model
m = Sequential([
    SimpleRNN(8, input_shape=(1,4)),
    Dense(4, activation='softmax')
])
m.compile('adam','categorical_crossentropy')
m.fit(X,y,epochs=100,verbose=0)

# Test all
for i,c in enumerate(chars[:-1]):
    pred = m.predict(np.eye(4)[i].reshape(1,1,4), verbose=0)
    print(c, "->", chars[np.argmax(pred)])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


a -> b
b -> c
c -> d


### BERT

In [ ]:
# Implement BERT model# install (only if needed)
!pip install -q transformers

from transformers import pipeline

# load BERT-based sentiment model
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

# input
text = "I love this product!"

# predict
result = classifier(text)[0]

# binary output
binary_output = 1 if result['label'] == "POSITIVE" else 0

print("Text:", text)
print("Model Output:", result)
print("Binary:", binary_output)

### GAN

In [ ]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# data
(x_train, _), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.reshape(-1, 784) / 255.0

# generator
generator = Sequential([
    Dense(128, activation='relu', input_dim=100),
    Dense(784, activation='sigmoid')])

# discriminator
discriminator = Sequential([
    Dense(128, activation='relu', input_dim=784),
    Dense(1, activation='sigmoid')])

discriminator.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])


# --- Train GAN ---
noise = np.random.normal(0, 1, (128, 100))          # Random noise
fake_images = generator.predict(noise)              # Generate fake images

real_images = x_train[np.random.randint(0, x_train.shape[0], 128)]  # Sample real images

# Labels for training
real_labels = np.ones((128, 1))
fake_labels = np.zeros((128, 1))

# Train discriminator
d_loss_real = discriminator.train_on_batch(real_images, real_labels)
d_loss_fake = discriminator.train_on_batch(fake_images, fake_labels)

# Print discriminator performance
print("Discriminator accuracy on real:", d_loss_real[1]*100)
print("Discriminator accuracy on fake:", d_loss_fake[1]*100)